In [1]:
!pip install pymupdf sentence-transformers

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import fitz
import os
import json
from sentence_transformers import SentenceTransformer


C:\Users\vasu mittal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


load pdf


In [3]:
pdf_path = "digitalbook.pdf"
doc=fitz.open(pdf_path)
print ("Total pages:", len(doc))


Total pages: 565


In [4]:
pages = []
for page_number,page in enumerate(doc):
    text = page.get_text()
    pages.append({"page_number": page_number, "text": text})    
print(pages[0])  # Print the first page's content


{'page_number': 0, 'text': ''}


TEXT CLEANING

In [5]:
def clean_text(text):
    # Remove newlines and extra spaces
     text = text.replace('\n', ' ')
     text = ' '.join(text.split())
   
     return text

In [6]:
for page in pages:
    page['text'] = clean_text(page['text'])

print(pages[4])  # Print the cleaned first page's content




{'page_number': 4, 'text': 'Digital Design With an Introduction to the Verilog HDL FIFTH EDITION M. Morris Mano Emeritus Professor of Computer Engineering California State University, Los Angeles Michael D. Ciletti Emeritus Professor of Electrical and Computer Engineering University of Colorado at Colorado Springs Upper Saddle River Boston Columbus San Franciso New York Indianapolis London Toronto Sydney Singapore Tokyo Montreal Dubai Madrid Hong Kong Mexico City Munich Paris Amsterdam Cape Town'}


CHUNKING

In [7]:
def chunk_code(text, chunk_size=2000, overlap=200):
    """
    Splits the text into chunks of specified size with a specified overlap.
    
    Parameters:
    - text (str): The input text to be chunked.
    - chunk_size (int): The maximum size of each chunk.
    - overlap (int): The number of overlapping characters between chunks.
    
    Returns:
    - List[str]: A list of text chunks.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap  # Move start forward by chunk_size minus overlap
    return chunks


In [8]:
chunks = []
for page in pages:
    page_chunks = chunk_code(page['text'])
    for chunk in page_chunks:
        chunks.append({"page_number": page['page_number'], "chunk": chunk})

print("tottal chunks:", len(chunks))
print(chunks[3])  # Print the first chunk's content

tottal chunks: 906
{'page_number': 5, 'chunk': "Library of Congress Cataloging-in-Publication Data Mano, M. Morris, 1927– Digital design : with an introduction to the verilog hdl / M. Morris Mano, Michael D. Ciletti.—5th ed. p. cm. Includes index. ISBN-13: 978-0-13-277420-8 ISBN-10: 0-13-277420-8 1. Electronic digital computers—Circuits. 2. Logic circuits. 3. Logic design. 4. Digital integrated circuits. I. Ciletti, Michael D. II. Title. TK7888.3.M343 2011 621.39'5—dc23 2011039094 Vice President and Editorial Director, ECS: Marcia J. Horton Executive Editor: Andrew Gilfillan Vice-President, Production: Vince O’Brien Executive Marketing Manager: Tim Galligan Marketing Assistant: Jon Bryant Permissions Project Manager: Karen Sanatar Senior Managing Editor: Scott Disanno Production Project Manager/Editorial Production Manager: Greg Dulles Cover Designer: Jayne Conte Cover Photo: Michael D. Ciletti Composition: Jouve India Private Limited Full-Service Project Management: Jouve India Privat

EMBEDDING

In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 651.80it/s]


In [10]:
texts = [chunk['chunk'] for chunk in chunks]
embeddings = model.encode(texts,show_progress_bar=True)


Batches: 100%|██████████| 29/29 [00:44<00:00,  1.54s/it]


In [11]:
print("Embeddings shape:", embeddings.shape)  # Print the shape of the embeddings


Embeddings shape: (906, 384)


In [12]:
for i, chunk in enumerate(chunks):
    chunk['embedding'] = embeddings[i].tolist()  # Convert numpy array to list for JSON serialization

SAVING THE PROCESSED DATA

In [13]:
with open('chunks_with_embeddings.json', 'w') as f:
    json.dump(chunks, f, indent=4)

STORE EMBEDDING IN VECTORED DATABASE

In [14]:
!pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [15]:
import faiss
import numpy as np

In [16]:
embedding_matrix = np.array([chunk['embedding'] for chunk in chunks]).astype('float32')
print("Embedding matrix shape:", embedding_matrix.shape)  # Print the shape of the embedding matrix

Embedding matrix shape: (906, 384)


In [17]:
dim = embedding_matrix.shape[1]  # Dimension of the embeddings
index = faiss.IndexFlatL2(dim)  # Create a FAISS index for L2 distance
index.add(embedding_matrix)  # Add the embeddings to the index  
print("Number of vectors in the index:", index.ntotal)  # Print the number of vectors in the index

Number of vectors in the index: 906


SEMANTIC SEARCH


In [18]:
def semantic_search(query, top_k=5):
    """
    Perform semantic search using the existing FAISS index.
    Returns the most relevant chunks with their distances.
    """

    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    if len(chunks) == 0:
        return []

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    actual_k = min(top_k, len(chunks))

    distances, indices = index.search(
        query_embedding,
        actual_k
    )

    results = []

    for idx, distance in zip(indices[0], distances[0]):

        # Ignore invalid FAISS results.
        if idx < 0 or idx >= len(chunks):
            continue

        results.append({
            "chunk_index": int(idx),
            "page_number": chunks[idx]["page_number"] + 1,
            "chunk": chunks[idx]["chunk"],
            "distance": float(distance)
        })

    return results


In [19]:
from sentence_transformers import SentenceTransformer

test_query = "What is the main topic of the book?"
# Ensure the embedding model is available before running semantic search
if 'model' not in globals():
    model = SentenceTransformer('all-MiniLM-L6-v2')

if 'index' not in globals():
    if 'embedding_matrix' not in globals():
        embedding_matrix = np.asarray(
            [chunk['embedding'] for chunk in chunks],
            dtype='float32'
        )
    index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    index.add(embedding_matrix)

results = semantic_search(test_query, top_k=5)
for result in results:
    print(f"Page Number: {result['page_number']}, Distance: {result['distance']}")
    print(f"Chunk: {result['chunk']}\n")    

Page Number: 86, Distance: 1.298074722290039
Chunk: veral exercises and design problems based on Verilog throughout the book.

Page Number: 367, Distance: 1.576037049293518
Chunk: Design. Upper Saddle River, NJ: Prentice Hall.

Page Number: 129, Distance: 1.6590080261230469
Chunk: t is a comment (optional) providing useful information to the reader. The second line begins with the keyword module and starts the declaration (description) of the module; the last line completes the declaration with the keyword endmodule. The keyword module is followed by a name and a list of ports. The name ( Simple_Circuit in this example) is an identifier. Identifiers are names given to modules, variables (e.g., a signal), and other elements of the language so that they can be referenced in the design. In general, we choose meaningful names for modules. Identifiers are composed of alpha- numeric characters and the underscore (_), and are case sensitive. Identifiers must start with an alphabetic character

In [20]:
!pip install rank-bm25

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [21]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r"\w+", text.lower())


In [22]:
corpus = [chunk["chunk"] for chunk in chunks]

tokenized_corpus = [
    tokenize(text)
    for text in corpus
]

bm25 = BM25Okapi(tokenized_corpus)

query = "What is the main topic of the book?"
tokenized_query = tokenize(query)
scores = bm25.get_scores(tokenized_query)
top_indices = scores.argsort()[-5:][::-1]
for idx in top_indices:
    print(f"Page Number: {chunks[idx]['page_number'] + 1}, Score: {scores[idx]}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")


Page Number: 267, Score: 18.94256823116579
Chunk: Problems 249 (a)* Use D flip-flops in the design. (b) Use JK flip-flops in the design. 001 010 011 100 000 1/1 1/1 1/0 1/1 1/1 0/0 0/0 0/0 0/0 0/0 FIGURE P5.19 5.20 Design the sequential circuit specified by the state diagram of Fig. 5.19 , using T flip-flops. 5.21 What is the main difference between an initial statement and an always statement in Verilog HDL? 5.22 Draw the waveform generated by the statements below: (a) initial begin w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; end (b) initial fork w = 0; #10 w = 1; # 40 w = 0; # 20 w = 1; #15 w = 0; join 5.23* Consider the following statements assuming that RegA contains the value of 50 initially. (a) RegA = 125; (b) RegA <= 125; RegB = RegA; RegB <= RegA; What are the values of RegA and RegB after execution? 5.24 Write and verify an HDL behavioral description of a positive-edge-sensitive D flip-flop with asynchronous preset and clear. 5.25 A special positive-edge-triggered fl

In [23]:
# Map semantic results back to their chunk indices
semantic_rank = {
    item["chunk_index"]: rank
    for rank, item in enumerate(results, start=1)
}

bm25_rank = {int(idx): rank for rank, idx in enumerate(top_indices, start=1)}

# Reciprocal rank fusion; lower ranks are better
candidate_indices = set(semantic_rank) | set(bm25_rank)
final_scores = {
    idx: 1 / (60 + semantic_rank.get(idx, 1000))
       + 1 / (60 + bm25_rank.get(idx, 1000))
    for idx in candidate_indices
}

for idx in sorted(candidate_indices, key=final_scores.get, reverse=True)[:5]:
    print(f"Page Number: {chunks[idx]['page_number'] + 1}")
    print(f"Chunk: {chunks[idx]['chunk']}\n")

print("Final Scores:", final_scores)  # Print the final scores for debugging


Page Number: 12
Chunk: x Preface (HDLs), and this edition of the book gives greater attention to their use and presents what we think is a clear development of a design methodology using the Verilog HDL. MULT I ‐M ODA L L E A R N IN G Digital Design supports a multimodal approach to learning. The so‐called VARK char- acterization of learning modalities identifies four major modes by which humans learn: (V) visual, (A) aural, (R) reading, and (K) kinesthetic. In hindsight, we note that the relatively high level of illustrations and graphical content of our text addresses the visual (V) component of VARK; discussions and numerous examples address the reading (R) component. Students who exploit the availability of free simulators to work assignments are led through a kinesthetic (K) learning experience, including the positive feedback and delight of designing a logic system that works. The remaining element of VARK, the aural/auditory (A) experience, is left to the instructor. We have pro

In [24]:
from sentence_transformers import cross_encoder

# Initialize the cross-encoder for semantic rankingre
reranker = cross_encoder.CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1079.60it/s]


In [25]:
!pip install ollama
import ollama

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [27]:
try:
    response = ollama.chat(
        model="llama3.1:8b",
        messages=[
            {
                "role": "user",
                "content": "Explain RAG in simple terms."
            }
        ],
        options={
            "num_gpu": 0,
            "num_ctx": 1024,
            "num_batch": 16,
            "num_predict": 200,
            "temperature": 0
        },
        keep_alive="1m"
    )
except Exception as exc:
    response = {
        "message": {
            "content": f"Ollama failed: {exc}"
        }
    }

print(response["message"]["content"])

RAG is a simple concept that stands for "Red, Amber, Green". It's a way to categorize and track the status of tasks, projects, or processes. Here's a breakdown of what each color means:

1. **Red (R)**: This indicates a problem or an issue that needs to be addressed immediately. It's like a warning sign that says "Something's not right, fix it now!"
2. **Amber (A)**: This color represents a warning or a potential issue that needs attention soon. It's like a caution sign that says "Be aware, something might go wrong if you don't take action."
3. **Green (G)**: This is the good news! Green indicates that everything is okay, and the task or project is on track. It's like a thumbs-up sign that says "You're doing great, keep going!"

RAG is often used in business, project management, and IT to:

* Track progress and identify potential


In [28]:
import ollama

def generate_with_ollama(
    prompt,
    num_ctx=1024,
    num_predict=200,
    temperature=0
):
    try:
        response = ollama.chat(
            model="llama3.1:8b",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "num_ctx": num_ctx,
                "num_predict": num_predict,
                "num_batch": 64,
                "num_gpu": 0,
                "temperature": temperature
            },
            keep_alive="1m"
        )

        return response["message"]["content"].strip()

    except Exception as exc:
        print("Ollama request failed.")
        print("Error:", str(exc))
        return None

# Test helper function
test_answer = generate_with_ollama(
    "In AI, what does RAG stand for? Explain Retrieval-Augmented Generation in 2 simple sentences.",
    num_ctx=1024,
    num_predict=200,
    temperature=0
)

if test_answer is None:
    print("Unable to generate an answer. Check Ollama and available memory.")
else:
    print(test_answer)


In AI, RAG stands for Retrieval-Augmented Generation.

RAG is a type of AI model that combines the strengths of two techniques: retrieval and generation. It first retrieves relevant information from a large database or knowledge base, and then uses this information to inform and augment the generation of new text or responses.


In [29]:
candidate_order = sorted(
    candidate_indices,
    key=final_scores.get,
    reverse=True
)[:10]

print("Number of candidates:", len(candidate_order))

pairs = [
    (test_query, chunks[idx]["chunk"])
    for idx in candidate_order
]

print("Pairs created:", len(pairs))

rerank_scores = reranker.predict(pairs)

reranked_results = sorted(
    zip(candidate_order, rerank_scores),
    key=lambda item: item[1],
    reverse=True
)

print("Reranking complete.")

for idx, score in reranked_results[:5]:
    print("Page:", chunks[idx]["page_number"])
    print("Reranker score:", float(score))
    print("Text:", chunks[idx]["chunk"][:500])
    print("-" * 80)

Number of candidates: 9
Pairs created: 9
Reranking complete.
Page: 11
Reranker score: -7.482226371765137
Text: x Preface (HDLs), and this edition of the book gives greater attention to their use and presents what we think is a clear development of a design methodology using the Verilog HDL. MULT I ‐M ODA L L E A R N IN G Digital Design supports a multimodal approach to learning. The so‐called VARK char- acterization of learning modalities identifies four major modes by which humans learn: (V) visual, (A) aural, (R) reading, and (K) kinesthetic. In hindsight, we note that the relatively high level of illu
--------------------------------------------------------------------------------
Page: 85
Reranker score: -8.129461288452148
Text: veral exercises and design problems based on Verilog throughout the book.
--------------------------------------------------------------------------------
Page: 20
Reranker score: -9.184770584106445
Text: ilt. It is also used in conjunction with logic syn- 

In [30]:
def ask_documind(question, retrieval_k=15, doc_filter=None):

    if doc_filter:
        retrieval_k = max(retrieval_k, 50)
        if isinstance(doc_filter, str):
            doc_filter = [doc_filter]

    # 1. Semantic search
    semantic_results = semantic_search(
        question,
        top_k=retrieval_k
    )

    semantic_rank = {
        item["chunk_index"]: rank
        for rank, item in enumerate(semantic_results, start=1)
    }

    # 2. BM25 keyword search
    tokenized_query = tokenize(question)

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_indices = bm25_scores.argsort()[-retrieval_k:][::-1]

    bm25_rank = {
        int(idx): rank
        for rank, idx in enumerate(bm25_indices, start=1)
    }

    # 3. Reciprocal Rank Fusion & document filtering
    candidate_indices = set(semantic_rank) | set(bm25_rank)
    if doc_filter:
        candidate_indices = {i for i in candidate_indices if chunks[i].get("doc") in doc_filter}

    if not candidate_indices:
        return {
            "question": question,
            "answer": "Information not found in the provided document.",
            "citation_validation": {"cited_sources": [], "is_valid": True, "invalid_sources": []},
            "source_metadata": {},
            "retrieved_chunks": []
        }

    final_scores = {
        idx: (
            1 / (60 + semantic_rank.get(idx, 1000))
            + 1 / (60 + bm25_rank.get(idx, 1000))
        )
        for idx in candidate_indices
    }

    candidate_order = sorted(
        candidate_indices,
        key=final_scores.get,
        reverse=True
    )[:15]

    # 4. Neural reranking
    pairs = [
        (question, chunks[idx]["chunk"])
        for idx in candidate_order
    ]

    rerank_scores = reranker.predict(pairs)

    reranked_results = sorted(
        zip(candidate_order, rerank_scores),
        key=lambda item: item[1],
        reverse=True
    )

    # 5. Select the top 3 chunks to prevent prompt context overflow
    top_results = reranked_results[:3]

    # 6. Build context with source IDs and doc & page metadata
    context_parts = []
    source_metadata = {}

    for source_number, (idx, score) in enumerate(
        top_results,
        start=1
    ):
        chunk = chunks[idx]
        source_id = f"S{source_number}"

        page_label = chunk["page_number"] + 1
        doc_name = chunk.get("doc", "document")

        source_metadata[source_id] = {
            "chunk_index": idx,
            "doc": doc_name,
            "page_number": page_label,
            "reranker_score": float(score),
            "text": chunk["chunk"]
        }

        context_parts.append(
            f"[{source_id}] ({doc_name}, page {page_label})\n"
            f"{chunk['chunk']}"
        )

    context = "\n\n".join(context_parts)

    # 7. Create grounded prompt with citation rules
    prompt = f"""
You are DocuMind, an AI document intelligence assistant.

Answer the user's question using ONLY the information provided in the context below.

CITATION RULES:
- After each claim, cite its source with the tag from the context, like [S1] or [S2].
- Only use tags that appear in the context. Never invent tags or page numbers.
- If the answer is not in the context, say:
  "Information not found in the provided document."

If the answer is present in the context, answer it directly and clearly.
Do not say that the information is unavailable if the context contains the answer.

DOCUMENT CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

    # 8. Generate answer using Ollama
    answer = generate_with_ollama(
        prompt,
        num_ctx=2048,
        num_predict=250,
        temperature=0
    )

    if answer is None:
        answer = "Unable to generate an answer. Check Ollama and available memory."

    # 9. Citation validation
    def validate_citations(answer_text, valid_ids):
        cited_ids = set(re.findall(r"\[(S\d+)\]", answer_text))
        invalid_ids = cited_ids - set(valid_ids)
        return {
            "cited_sources": sorted(list(cited_ids)),
            "is_valid": len(invalid_ids) == 0,
            "invalid_sources": sorted(list(invalid_ids))
        }

    citation_validation = validate_citations(answer, list(source_metadata.keys()))

    return {
        "question": question,
        "answer": answer,
        "citation_validation": citation_validation,
        "source_metadata": source_metadata,
        "retrieved_chunks": [
            {
                "chunk_id": idx,
                "source_id": f"S{i}",
                "doc": chunks[idx].get("doc", "document"),
                "page_number": chunks[idx]["page_number"] + 1,
                "reranker_score": float(score),
                "text": chunks[idx]["chunk"]
            }
            for i, (idx, score) in enumerate(top_results, start=1)
        ]
    }


In [31]:
result = ask_documind(
    "How does a D flip-flop work?"
)

print("QUESTION:")
print(result["question"])

print("\nANSWER:")
print(result["answer"])

if "citation_validation" in result:
    print("\nCITATION VALIDATION:")
    print("Valid citations:", result["citation_validation"]["is_valid"])
    print("Cited sources:", result["citation_validation"]["cited_sources"])

print("\nSOURCES:")
for source in result["retrieved_chunks"]:
    print(
        f"[{source.get('source_id', 'S')}] Page: {source['page_number']} | "
        f"Reranker score: {source['reranker_score']:.4f}"
    )


QUESTION:
How does a D flip-flop work?

ANSWER:
A D flip-flop can set or reset the output, depending on the value of the D input immediately before the clock transition. [S1]

CITATION VALIDATION:
Valid citations: True
Cited sources: ['S1']

SOURCES:
[S1] Page: 218 | Reranker score: 5.6063
[S2] Page: 349 | Reranker score: 5.3011
[S3] Page: 219 | Reranker score: 5.0505


SUMMARISATION

In [32]:
import ollama
import re

def summarise_chunk_batch(batch, summary_type="detailed"):
    context_parts = []
    for chunk in batch:
        page_number = chunk.get("page_label")
        if page_number is None:
            page_number = chunk.get("page_number", "unknown")
            if isinstance(page_number, int):
                page_number += 1
        page_text = chunk.get("chunk", "")
        doc_name = chunk.get("doc", "document")
        context_parts.append(f"[{doc_name} | Page {page_number}]:\n{page_text}")
    context = "\n\n".join(context_parts)
    summary_instruction = {
        "short": "Summarize the following text in a concise manner, highlighting the key points and main ideas.",
        "detailed": "Provide a detailed summary of the following text, including all relevant information and important details.",
        "bullet points": "Summarize the following text in bullet points, focusing on the most important aspects and key takeaways."
    }
    normalized_summary_type = summary_type.lower().replace(" summary", "")
    instruction = summary_instruction.get(normalized_summary_type, "Provide a detailed summary of the following text.")
    prompt = f"""You are DocuMind, an AI document intelligence assistant. Summarize the supplied document excerpts.

Instructions:
- {instruction}
- Use only information in the supplied text.
- Do not invent facts or introduce outside information.
- Preserve important technical terms.
- Include PDF page references for important claims.
- If the excerpts contain insufficient information, state that clearly.
- Avoid repeating the same idea.

DOCUMENT EXCERPTS:
{context}

SUMMARY:
"""

    try:
        response = ollama.chat(
            model="llama3.1:8b",
            messages=[{"role": "user", "content": prompt}],
            options={
                "num_ctx": 2048,
                "num_predict": 250,
                "num_batch": 64,
                "num_gpu": 0,
                "temperature": 0.1
            },
            keep_alive="1m"
        )
        summary = response["message"]["content"].strip()
    except Exception as exc:
        print("Ollama request failed:", exc)
        summary = None

    return summary or "Unable to generate batch summary."


In [33]:
def summarize_document(
    chunks,
    summary_type="detailed",
    batch_size=1,
    max_chunks=10
):
    """
    Summarize a document using hierarchical map-reduce.

    max_chunks:
        10   = test with the first 10 chunks
        None = process all chunks
    """

    if not chunks:
        raise ValueError("No document chunks are available.")

    if batch_size < 1:
        raise ValueError("batch_size must be at least 1.")

    # Use a limited number of chunks during initial testing
    selected_chunks = (
        chunks[:max_chunks]
        if max_chunks is not None
        else chunks
    )

    intermediate_summaries = []

    # MAP: Summarize each batch
    for start in range(0, len(selected_chunks), batch_size):

        batch = selected_chunks[start:start + batch_size]

        start_page = batch[0].get(
            "page_label",
            batch[0]["page_number"] + 1
        )

        end_page = batch[-1].get(
            "page_label",
            batch[-1]["page_number"] + 1
        )

        print(
            f"Summarizing batch "
            f"{start // batch_size + 1} "
            f"(approximately pages {start_page}-{end_page})..."
        )

        batch_summary = summarise_chunk_batch(
            batch,
            summary_type=summary_type
        )

        intermediate_summaries.append({
            "start_page": start_page,
            "end_page": end_page,
            "summary": batch_summary
        })

    # Hierarchical REDUCE: Combine intermediate summaries progressively if many
    current_summaries = intermediate_summaries
    while len(current_summaries) > 4:
        next_level = []
        reduce_batch_size = 3
        for i in range(0, len(current_summaries), reduce_batch_size):
            group = current_summaries[i:i + reduce_batch_size]
            comb = "\n\n".join(
                f"Source pages {item['start_page']}-{item['end_page']}:\n{item['summary']}"
                for item in group
            )
            hier_prompt = f"""You are DocuMind, an AI document summarization assistant.
Combine these intermediate section summaries into a single coherent summary.
Preserve key technical facts and page references.

SUMMARIES:
{comb}

COMBINED SUMMARY:
"""
            try:
                resp = ollama.chat(
                    model="llama3.1:8b",
                    messages=[{"role": "user", "content": hier_prompt}],
                    options={
                        "num_ctx": 2048,
                        "num_predict": 250,
                        "num_batch": 64,
                        "num_gpu": 0,
                        "temperature": 0.1
                    },
                    keep_alive="1m"
                )
                reduced = resp["message"]["content"].strip()
            except Exception as exc:
                print("Ollama request failed:", exc)
                reduced = None

            next_level.append({
                "start_page": group[0]["start_page"],
                "end_page": group[-1]["end_page"],
                "summary": reduced or comb
            })
        current_summaries = next_level

    # Final reduction
    combined_summaries = "\n\n".join(
        (
            f"Source pages {item['start_page']}-"
            f"{item['end_page']}:\n"
            f"{item['summary']}"
        )
        for item in current_summaries
    )

    final_prompt = f"""
You are DocuMind, an AI document summarization assistant.

Combine the following intermediate summaries into one coherent summary.

Requirements:
- {summary_type.replace("_", " ").capitalize()} summary.
- Preserve the main ideas and important technical details.
- Remove repetition.
- Do not introduce facts that are absent from the intermediate summaries.
- Retain PDF page references associated with the claims.
- If a claim cannot be traced to the supplied summaries, omit it.

INTERMEDIATE SUMMARIES:
{combined_summaries}

FINAL SUMMARY:
"""

    try:
        response = ollama.chat(
            model="llama3.1:8b",
            messages=[
                {
                    "role": "user",
                    "content": final_prompt
                }
            ],
            options={
                "num_ctx": 2048,
                "num_predict": 250,
                "num_batch": 64,
                "num_gpu": 0,
                "temperature": 0.1
            },
            keep_alive="1m"
        )
        final_summary = response["message"]["content"].strip()
    except Exception as exc:
        print("Ollama request failed:", exc)
        final_summary = None

    if final_summary is None:
        final_summary = "Unable to generate summary. Check Ollama and available memory."

    return {
        "summary": final_summary,
        "chunks_processed": len(selected_chunks),
        "intermediate_summaries": intermediate_summaries
    }


OCR

In [34]:
!pip install easyocr


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------- ------------------------- 1.0/2.9 MB 5.0 MB/s eta 0:00:01
   ------------------------- -------------- 1.8/2.9 MB 4.0 MB/s eta 0:00:01
   ------------------------------------ --- 2.6/2.9 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 3.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------- ----------------- 0.8/1.4 MB 3.9 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 3.6 MB/s  0:00:00
   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.8/124.1 MB 4.9 MB/s eta 0:00:26
    --------------------------------------- 1.8/124.1 MB 4.2 MB/s eta 0:00:30
    --------------------------------------- 2.6/124.1 MB 4.1 MB/s eta 0:00:30
   - ------------------------

  You can safely remove it manually.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\vasu mittal\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [35]:
import easyocr

reader = easyocr.Reader(['en'], gpu=False)

def ocr_page(pdf_page, zoom=2):
    pix = pdf_page.get_pixmap(matrix=fitz.Matrix(zoom, zoom))
    return " ".join(reader.readtext(pix.tobytes("png"), detail=0, paragraph=True))

def extract_pages(pdf_path):
    out = []
    with fitz.open(pdf_path) as pdf:
        for n, p in enumerate(pdf):
            text = p.get_text().strip()
            if len(text) < 50:          # OCR only scanned pages
                text = ocr_page(p)
            out.append({"doc": os.path.basename(pdf_path),
                        "page_number": n,
                        "text": clean_text(text)})
    return out


Using CPU. Note: This module is much faster with a GPU.


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

C:\Users\vasu mittal\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [36]:
safe_chunks = [
    chunk for chunk in chunks
    if isinstance(chunk, dict)
    and isinstance(chunk.get("chunk"), str)
    and chunk["chunk"].strip()
]

doc_chunks = [c for c in safe_chunks if c.get("doc") == "digitalbook.pdf"]
section = [c for c in doc_chunks if 20 <= c.get("page_number", 0) < 24]

if not section:
    print("No valid section chunks found.")
else:
    print("Section chunks:", len(section), "| pages", section[0]["page_number"] + 1, "to", section[-1]["page_number"] + 1)
    summary_result = summarize_document(
        section,
        summary_type="short",
        batch_size=1,
        max_chunks=None
    )

    print("Final Summary:")
    print(summary_result["summary"])

    print("\nChunks processed:", summary_result["chunks_processed"])


No valid section chunks found.


In [37]:
import psutil

memory = psutil.virtual_memory()

print(f"Total RAM: {memory.total / (1024**3):.2f} GB")
print(f"Available RAM: {memory.available / (1024**3):.2f} GB")
print(f"RAM Used: {memory.percent}%")

Total RAM: 15.70 GB
Available RAM: 5.59 GB
RAM Used: 64.4%


MULTIDOCUMENT SUPPORT

In [38]:
CHUNKS_FILE = "chunks_with_embeddings.json"

def build_indices():
    global index, bm25, embedding_matrix
    embedding_matrix = np.array([c["embedding"] for c in chunks], dtype="float32")
    index = faiss.IndexFlatL2(embedding_matrix.shape[1])
    index.add(embedding_matrix)
    bm25 = BM25Okapi([tokenize(c["chunk"]) for c in chunks])

def add_documents(pdf_paths):
    global chunks
    known = {c.get("doc") for c in chunks}
    for path in pdf_paths:
        name = os.path.basename(path)
        if name in known:
            print("Already indexed:", name)
            continue
        new = [{"doc": name, "page_number": p["page_number"], "chunk": t}
               for p in extract_pages(path) for t in chunk_code(p["text"])]
        if not new:
            print("No text found:", name)
            continue
        embs = model.encode([c["chunk"] for c in new], show_progress_bar=True)
        for c, e in zip(new, embs):
            c["embedding"] = e.tolist()
        chunks.extend(new)
        print(f"Indexed {name}: {len(new)} chunks")
    with open(CHUNKS_FILE, "w") as f:
        json.dump(chunks, f)
    build_indices()

def remove_document(name):
    global chunks
    chunks = [c for c in chunks if c.get("doc") != name]
    with open(CHUNKS_FILE, "w") as f:
        json.dump(chunks, f)
    if chunks:
        build_indices()

def list_documents():
    return sorted({c.get("doc", "unknown") for c in chunks})

if os.path.exists(CHUNKS_FILE):
    with open(CHUNKS_FILE) as f:
        chunks = json.load(f)
    for c in chunks:
        c.setdefault("doc", "digitalbook.pdf")   # old chunks had no doc field
    build_indices()
else:
    chunks = []
